# GeoGuessr Country Classifier — Evaluation

This notebook loads the trained v15 model weights and evaluates performance on the held-out test set.

**Metrics computed:**
- Top-1 accuracy (primary metric)
- Top-5 accuracy
- Per-class accuracy
- Confusion matrix
- Sample predictions with confidence scores

In [ ]:
%pip install torch torchvision kagglehub pillow matplotlib seaborn numpy --quiet

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from collections import defaultdict
from torch.utils.data import DataLoader
from torchvision import transforms

# Import from the geoguessr package
import sys
sys.path.append('..')
from geoguessr.model import GeoConvModelV1
from geoguessr.dataset import build_merged_dataset, make_dataloaders, eval_transform, MergedGeoDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 1. Load Dataset

In [ ]:
import kagglehub

geo_path = kagglehub.dataset_download("ubitquitin/geolocation-geoguessr-images-50k")
gsv_path = kagglehub.dataset_download("amaralibey/gsv-cities")

data_root       = os.path.join(geo_path, "compressed_dataset")
gsv_images_path = os.path.join(gsv_path, "Images")

all_samples, classes, country_to_idx = build_merged_dataset(data_root, gsv_images_path, random_seed=42)
num_classes = len(classes)

_, _, test_loader = make_dataloaders(all_samples, random_seed=42)
print(f"\nTest set ready | {num_classes} classes")

## 2. Load Pretrained Weights

In [ ]:
MODEL_PATH = "/gpfs/projects/dsci410_510/Rylan_K/geo_cnn_weights_v15_min100.pth"

checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model = GeoConvModelV1(num_classes=checkpoint["num_classes"]).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Loaded: {MODEL_PATH}")
print(f"Classes    : {checkpoint['num_classes']}")
print(f"Saved acc  : {checkpoint.get('accuracy', 'unknown'):.4f}")

## 3. Top-1 and Top-5 Accuracy

In [ ]:
def evaluate(model, loader, device, top_k=(1, 5)):
    model.eval()
    correct = {k: 0 for k in top_k}
    total = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            for k in top_k:
                topk_preds = torch.topk(logits, k, dim=1).indices
                correct[k] += (topk_preds == y.unsqueeze(1)).any(dim=1).sum().item()
            all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            total += y.size(0)

    accs = {k: correct[k] / total for k in top_k}
    return accs, np.array(all_preds), np.array(all_labels)

accs, all_preds, all_labels = evaluate(model, test_loader, device)

print("=" * 40)
print(f"  Top-1 Accuracy : {accs[1]*100:.2f}%")
print(f"  Top-5 Accuracy : {accs[5]*100:.2f}%")
print(f"  Test samples   : {len(all_labels)}")
print(f"  Random baseline: {100/num_classes:.2f}%")
print("=" * 40)

## 4. Per-Class Accuracy

In [ ]:
per_class_correct = defaultdict(int)
per_class_total   = defaultdict(int)

for pred, label in zip(all_preds, all_labels):
    per_class_total[label] += 1
    if pred == label:
        per_class_correct[label] += 1

per_class_acc = {
    classes[i]: per_class_correct[i] / per_class_total[i]
    for i in range(num_classes) if per_class_total[i] > 0
}

sorted_acc = sorted(per_class_acc.items(), key=lambda x: x[1], reverse=True)

print("Top 10 best predicted countries:")
for country, acc in sorted_acc[:10]:
    print(f"  {country:30s}  {acc*100:.1f}%")

print("\nBottom 10 hardest countries:")
for country, acc in sorted_acc[-10:]:
    print(f"  {country:30s}  {acc*100:.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
countries_sorted = [x[0] for x in sorted_acc]
accs_sorted      = [x[1] for x in sorted_acc]

colors = ['#2ecc71' if a >= 0.5 else '#e67e22' if a >= 0.25 else '#e74c3c' for a in accs_sorted]
bars = ax.barh(countries_sorted, [a * 100 for a in accs_sorted], color=colors)

ax.axvline(x=100/num_classes, color='black', linestyle='--', linewidth=1.5, label=f'Random baseline ({100/num_classes:.1f}%)')
ax.set_xlabel('Top-1 Accuracy (%)', fontsize=12)
ax.set_title('Per-Class Accuracy — GeoConvModelV1 (v15)', fontsize=14)
ax.legend()

green_patch  = mpatches.Patch(color='#2ecc71', label='≥ 50%')
orange_patch = mpatches.Patch(color='#e67e22', label='25–50%')
red_patch    = mpatches.Patch(color='#e74c3c', label='< 25%')
ax.legend(handles=[green_patch, orange_patch, red_patch, ax.lines[0]], fontsize=10)

plt.tight_layout()
plt.savefig('per_class_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved per_class_accuracy.png")

## 5. Confusion Matrix (Top 20 Countries by Sample Count)

In [ ]:
from sklearn.metrics import confusion_matrix

# Use only the 20 most-sampled countries for readability
top20_idx = sorted(per_class_total, key=lambda i: per_class_total[i], reverse=True)[:20]
top20_names = [classes[i] for i in top20_idx]

mask = np.isin(all_labels, top20_idx)
labels_sub = all_labels[mask]
preds_sub  = all_preds[mask]

# Remap to 0..19
remap = {old: new for new, old in enumerate(top20_idx)}
labels_sub = np.array([remap[l] for l in labels_sub])
preds_sub  = np.array([remap[p] if p in remap else -1 for p in preds_sub])

cm = confusion_matrix(labels_sub, preds_sub, labels=list(range(20)))
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm_norm, annot=False, fmt='.2f', cmap='Blues',
            xticklabels=top20_names, yticklabels=top20_names, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title('Normalized Confusion Matrix — Top 20 Countries', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved confusion_matrix.png")

## 6. Sample Predictions

Visualize individual test images alongside the model's top-5 predicted countries and confidence scores.

In [ ]:
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

# Grab a batch from the test set
test_samples_list = [all_samples[i] for i in range(len(all_samples))]

# Rebuild test indices the same way make_dataloaders does
np.random.seed(42)
n = len(all_samples)
indices = list(range(n))
np.random.shuffle(indices)
test_split = int(np.floor(0.15 * n))
test_idx   = indices[:test_split]
test_samples = [all_samples[i] for i in test_idx]

# Pick 8 random samples
np.random.seed(0)
sample_indices = np.random.choice(len(test_samples), 8, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for ax, idx in zip(axes.flat, sample_indices):
    img_path, true_label = test_samples[idx]
    img = Image.open(img_path).convert("RGB")
    tensor = eval_transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0]

    top5_probs, top5_idx = probs.topk(5)
    top5_countries = [classes[i] for i in top5_idx.cpu().numpy()]
    top5_confs     = top5_probs.cpu().numpy()
    true_country   = classes[true_label]
    correct        = (top5_idx[0].item() == true_label)

    ax.imshow(img.resize((224, 224)))
    ax.axis('off')

    pred_text = "\n".join([f"{'✓' if c == true_country else ' '} {c[:20]:20s} {p*100:.1f}%"
                             for c, p in zip(top5_countries, top5_confs)])
    color = '#2ecc71' if correct else '#e74c3c'
    ax.set_title(f"True: {true_country}\n{pred_text}", fontsize=7,
                 bbox=dict(boxstyle='round', facecolor=color, alpha=0.15))

plt.suptitle('Sample Test Predictions — Top-5 Confidence Scores\n(✓ = true label, green = correct top-1, red = incorrect)', fontsize=13)
plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved sample_predictions.png")

## 7. Summary

In [ ]:
print("=" * 50)
print("  EVALUATION SUMMARY — GeoConvModelV1 v15")
print("=" * 50)
print(f"  Classes         : {num_classes}")
print(f"  Test samples    : {len(all_labels)}")
print(f"  Top-1 Accuracy  : {accs[1]*100:.2f}%")
print(f"  Top-5 Accuracy  : {accs[5]*100:.2f}%")
print(f"  Random baseline : {100/num_classes:.2f}%")
print(f"  Improvement     : {accs[1]/(1/num_classes):.1f}x over random")
print("=" * 50)

above_50 = sum(1 for a in per_class_acc.values() if a >= 0.5)
above_25 = sum(1 for a in per_class_acc.values() if a >= 0.25)
print(f"\n  Countries >= 50% accuracy : {above_50} / {num_classes}")
print(f"  Countries >= 25% accuracy : {above_25} / {num_classes}")
print(f"  Best country  : {sorted_acc[0][0]:30s} {sorted_acc[0][1]*100:.1f}%")
print(f"  Worst country : {sorted_acc[-1][0]:30s} {sorted_acc[-1][1]*100:.1f}%")